# Libary Management #

In [1]:
#Libraries and dependencies 
import cv2
import cc3d
import math
import mrcfile
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import imagecodecs

import time
import tifffile as tiff
from PIL import Image

import scipy as scipy
from scipy.ndimage import center_of_mass, sum as ndimage_sum
from scipy.stats import linregress, kurtosis, mode
from scipy import ndimage
from scipy.ndimage import map_coordinates

from sklearn.preprocessing import StandardScaler

In [2]:
from skimage.restoration import denoise_bilateral
from skimage.util import img_as_float
import matplotlib as mpl

C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated and will be removed in a future release
  "class": algorithms.Blowfish,


In [3]:
#Use functions(openmrc, Ves_Analysis, df_forCluster) and libraries from previous notebook

import nbimporter
%run Vesicle_Analysis_Pipeline_Functions.ipynb

In [4]:
#For file pattern recognition and management

import os
import glob

In [5]:
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import matplotlib.patches as patches

# Aggregated Datasets #

**Normalized Distance Values** 

In [250]:
#Aggregated dataset
All_Unnorm_Ves_df = pd.concat([Unnorm_Ves_df, Unnorm_Ves_Glu_df, Unnorm_Ves_GIP_df, Unnorm_Ves_TAK_df, Unnorm_Ves_GKA_df, Unnorm_Ves_GLM_df,
                              Unnorm_Ves_Ext_df])
All_Unnorm_Ves_df

,Centroid X,Centroid Y,Centroid Z,Group,LAC Value,LAC Max,LAC Min,LAC Std Dev,LAC Skew,LAC Kurtosis,LAC Median,LAC 25th Quantile,LAC 75th Quantile,Volume (um3),Geometric Diameter (nm),Ellipsoid Surface Area (um2),Min Distance to PM EDT,Mito Dist EDT,Cell,Condition
0,48.910714,384.250000,229.500000,1,0.349779,0.401770,0.307330,0.020459,0.159612,-0.607521,0.349288,0.333847,0.364942,0.004537,173.087257,0.401855,0.029407,0.328711,Cell 1,No Stim
1,49.347826,416.076087,252.125000,2,0.322191,0.389055,0.268755,0.023362,0.397413,-0.173769,0.319054,0.306243,0.338172,0.004969,198.135448,0.500256,0.020466,0.299175,Cell 1,No Stim
2,51.742424,455.893939,238.515152,3,0.278407,0.316646,0.241714,0.013413,0.011086,0.630050,0.280071,0.270341,0.286476,0.001783,129.279009,0.211818,0.021089,0.311362,Cell 1,No Stim
3,52.241573,461.089888,218.522472,4,0.299589,0.349286,0.247234,0.018291,-0.055561,-0.038416,0.299293,0.286031,0.313549,0.004807,207.333430,0.585101,0.017665,0.277211,Cell 1,No Stim
4,53.133929,390.500000,222.785714,5,0.339127,0.391942,0.295986,0.019948,0.229311,-0.332529,0.337417,0.325317,0.353755,0.003025,159.414727,0.321185,0.045961,0.328880,Cell 1,No Stim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374,388.800000,159.683333,155.983333,375,0.355666,0.392179,0.317090,0.015680,0.067595,-0.182590,0.353680,0.343614,0.364843,0.002176,120.299278,0.184689,0.064769,0.057799,Cell 44,EXT
375,394.418605,163.279070,163.116279,376,0.343598,0.372269,0.321059,0.011730,0.214688,0.100983,0.344005,0.336139,0.351179,0.001560,105.091099,0.152246,0.094142,0.125265,Cell 44,EXT
376,397.461538,239.961538,202.384615,377,0.335783,0.370720,0.310605,0.015043,0.567027,-0.215087,0.332851,0.323903,0.343742,0.000943,86.750791,0.097556,0.326632,0.289696,Cell 44,EXT
377,401.350000,244.850000,210.700000,378,0.332563,0.363930,0.290191,0.016048,-0.780614,1.035219,0.334934,0.326027,0.341239,0.000725,75.783796,0.074144,0.189419,0.264104,Cell 44,EXT


In [254]:
#Filtered out large vesicles, and too correlated columns 
Filtered_All_Unnorm_Ves_df_drop = All_Unnorm_Ves_df.drop(['Centroid X','Centroid Y', 'Centroid Z', 'Group', 
                                      'LAC Max', 'LAC Min', 'LAC Median', 
                                       'LAC 25th Quantile', 'LAC 75th Quantile', 
                                       'Volume (um3)', 'Ellipsoid Surface Area (um2)'], axis = 1)
Filtered_All_Unnorm_Ves_df = Filtered_All_Unnorm_Ves_df_drop[
                            (Filtered_All_Unnorm_Ves_df_drop["Geometric Diameter (nm)"] < 600) &
                            (Filtered_All_Unnorm_Ves_df_drop["Geometric Diameter (nm)"] > 0)
                            ]

Filtered_All_Unnorm_Ves_df

,LAC Value,LAC Std Dev,LAC Skew,LAC Kurtosis,Geometric Diameter (nm),Min Distance to PM EDT,Mito Dist EDT,Cell,Condition
0,0.349779,0.020459,0.159612,-0.607521,173.087257,0.029407,0.328711,Cell 1,No Stim
1,0.322191,0.023362,0.397413,-0.173769,198.135448,0.020466,0.299175,Cell 1,No Stim
2,0.278407,0.013413,0.011086,0.630050,129.279009,0.021089,0.311362,Cell 1,No Stim
3,0.299589,0.018291,-0.055561,-0.038416,207.333430,0.017665,0.277211,Cell 1,No Stim
4,0.339127,0.019948,0.229311,-0.332529,159.414727,0.045961,0.328880,Cell 1,No Stim
...,...,...,...,...,...,...,...,...,...
374,0.355666,0.015680,0.067595,-0.182590,120.299278,0.064769,0.057799,Cell 44,EXT
375,0.343598,0.011730,0.214688,0.100983,105.091099,0.094142,0.125265,Cell 44,EXT
376,0.335783,0.015043,0.567027,-0.215087,86.750791,0.326632,0.289696,Cell 44,EXT
377,0.332563,0.016048,-0.780614,1.035219,75.783796,0.189419,0.264104,Cell 44,EXT


**Raw Values**

In [533]:
#Aggregated dataset for raw values 

All_Raw_Unnorm_Ves_df = pd.concat([Raw_Unnorm_Ves_df, Raw_Unnorm_VesGlu_df, Raw_Unnorm_Ves_GIP_df, Raw_Unnorm_Ves_TAK_df, Raw_Unnorm_Ves_GKA_df,
                                  Raw_Unnorm_Ves_GLM_df, Raw_Unnorm_Ves_Ext_df])
All_Raw_Unnorm_Ves_df

,Centroid X,Centroid Y,Centroid Z,Group,LAC Value,LAC Max,LAC Min,LAC Std Dev,LAC Skew,LAC Kurtosis,LAC Median,LAC 25th Quantile,LAC 75th Quantile,Volume (um3),Geometric Diameter (nm),Ellipsoid Surface Area (um2),Min Raw Distance to PM EDT,Mito Raw Dist EDT,Cell,Condition
0,48.910714,384.250000,229.500000,1,0.349779,0.401770,0.307330,0.020459,0.159612,-0.607521,0.349288,0.333847,0.364942,0.004537,173.087257,0.401855,73.492041,935.882026,Cell 1,No Stim
1,49.347826,416.076087,252.125000,2,0.322191,0.389055,0.268755,0.023362,0.397413,-0.173769,0.319054,0.306243,0.338172,0.004969,198.135448,0.500256,67.088748,851.789353,Cell 1,No Stim
2,51.742424,455.893939,238.515152,3,0.278407,0.316646,0.241714,0.013413,0.011086,0.630050,0.280071,0.270341,0.286476,0.001783,129.279009,0.211818,84.861300,886.485851,Cell 1,No Stim
3,52.241573,461.089888,218.522472,4,0.299589,0.349286,0.247234,0.018291,-0.055561,-0.038416,0.299293,0.286031,0.313549,0.004807,207.333430,0.585101,73.492041,789.255712,Cell 1,No Stim
4,53.133929,390.500000,222.785714,5,0.339127,0.391942,0.295986,0.019948,0.229311,-0.332529,0.337417,0.325317,0.353755,0.003025,159.414727,0.321185,120.012001,936.362828,Cell 1,No Stim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
370,388.800000,159.683333,155.983333,371,0.355666,0.392179,0.317090,0.015680,0.067595,-0.182590,0.353680,0.343614,0.364843,0.002176,120.299278,0.184689,93.625525,190.154341,Cell 44,EXT
371,394.418605,163.279070,163.116279,372,0.343598,0.372269,0.321059,0.011730,0.214688,0.100983,0.344005,0.336139,0.351179,0.001560,105.091099,0.152246,136.481484,412.111870,Cell 44,EXT
372,397.461538,239.961538,202.384615,373,0.335783,0.370720,0.310605,0.015043,0.567027,-0.215087,0.332851,0.323903,0.343742,0.000943,86.750791,0.097556,229.334764,953.073820,Cell 44,EXT
373,401.350000,244.850000,210.700000,374,0.332563,0.363930,0.290191,0.016048,-0.780614,1.035219,0.334934,0.326027,0.341239,0.000725,75.783796,0.074144,132.406488,868.878169,Cell 44,EXT


In [532]:
#Filtered out large vesicles, and too correlated columns 
Filtered_All_Raw_Unnorm_Ves_df_drop = All_Raw_Unnorm_Ves_df.drop(['Centroid X','Centroid Y', 'Centroid Z', 'Group', 
                                      'LAC Max', 'LAC Min', 'LAC Median', 
                                       'LAC 25th Quantile', 'LAC 75th Quantile', 
                                       'Volume (um3)', 'Ellipsoid Surface Area (um2)'], axis = 1)
Filtered_All_Raw_Unnorm_Ves_df = Filtered_All_Raw_Unnorm_Ves_df_drop[Filtered_All_Unnorm_Ves_df_drop["Geometric Diameter (nm)"] < 600]

Filtered_All_Raw_Unnorm_Ves_df

,LAC Value,LAC Std Dev,LAC Skew,LAC Kurtosis,Geometric Diameter (nm),Min Raw Distance to PM EDT,Mito Raw Dist EDT,Cell,Condition
0,0.349779,0.020459,0.159612,-0.607521,173.087257,73.492041,935.882026,Cell 1,No Stim
1,0.322191,0.023362,0.397413,-0.173769,198.135448,67.088748,851.789353,Cell 1,No Stim
2,0.278407,0.013413,0.011086,0.630050,129.279009,84.861300,886.485851,Cell 1,No Stim
3,0.299589,0.018291,-0.055561,-0.038416,207.333430,73.492041,789.255712,Cell 1,No Stim
4,0.339127,0.019948,0.229311,-0.332529,159.414727,120.012001,936.362828,Cell 1,No Stim
...,...,...,...,...,...,...,...,...,...
370,0.355666,0.015680,0.067595,-0.182590,120.299278,93.625525,190.154341,Cell 44,EXT
371,0.343598,0.011730,0.214688,0.100983,105.091099,136.481484,412.111870,Cell 44,EXT
372,0.335783,0.015043,0.567027,-0.215087,86.750791,229.334764,953.073820,Cell 44,EXT
373,0.332563,0.016048,-0.780614,1.035219,75.783796,132.406488,868.878169,Cell 44,EXT


# Working with Classes # 

**No stimulation**

In [173]:
#Performing Vesicle Analysis. Will take a while! 
Unnorm_VesNS_7_22 = NS_7_22_class.Ves_Analysis_unnorm()
Unnorm_VesNS_6_5 = NS_6_5_class.Ves_Analysis_unnorm()
Unnorm_VesNS_1537_16_18 = NS_1537_16_18_class.Ves_Analysis_unnorm()
Unnorm_VesNS_7_8_9 = NS_7_8_9_class.Ves_Analysis_unnorm()
Unnorm_VesNS_6_17_19 = NS_6_17_19_class.Ves_Analysis_unnorm()
Unnorm_VesNS_1537_19 = NS_1537_19_class.Ves_Analysis_unnorm()

C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang

In [175]:
cell1 = "Cell 1"
cell2 = "Cell 2"
cell3 = "Cell 3"
cell4 = "Cell 4"
cell5 = "Cell 5"
cell6 = "Cell 6"

Unnorm_VesNS_7_22["Cell"] = cell1
Unnorm_VesNS_6_5["Cell"] = cell2 
Unnorm_VesNS_1537_16_18["Cell"] = cell3
Unnorm_VesNS_7_8_9["Cell"] = cell4
Unnorm_VesNS_6_17_19["Cell"] = cell5 
Unnorm_VesNS_1537_19["Cell"] = cell6 

Unnorm_Ves_df = pd.concat([Unnorm_VesNS_7_22, Unnorm_VesNS_6_5, Unnorm_VesNS_1537_16_18, 
                           Unnorm_VesNS_7_8_9, Unnorm_VesNS_6_17_19, Unnorm_VesNS_1537_19])
Unnorm_Ves_df

,Centroid X,Centroid Y,Centroid Z,Group,LAC Value,LAC Max,LAC Min,LAC Std Dev,LAC Skew,LAC Kurtosis,LAC Median,LAC 25th Quantile,LAC 75th Quantile,Volume (um3),Geometric Diameter (nm),Ellipsoid Surface Area (um2),Min Distance to PM EDT,Mito Dist EDT,Cell
0,48.910714,384.250000,229.500000,1,0.349779,0.401770,0.307330,0.020459,0.159612,-0.607521,0.349288,0.333847,0.364942,0.004537,173.087257,0.401855,0.029407,0.328711,Cell 1
1,49.347826,416.076087,252.125000,2,0.322191,0.389055,0.268755,0.023362,0.397413,-0.173769,0.319054,0.306243,0.338172,0.004969,198.135448,0.500256,0.020466,0.299175,Cell 1
2,51.742424,455.893939,238.515152,3,0.278407,0.316646,0.241714,0.013413,0.011086,0.630050,0.280071,0.270341,0.286476,0.001783,129.279009,0.211818,0.021089,0.311362,Cell 1
3,52.241573,461.089888,218.522472,4,0.299589,0.349286,0.247234,0.018291,-0.055561,-0.038416,0.299293,0.286031,0.313549,0.004807,207.333430,0.585101,0.017665,0.277211,Cell 1
4,53.133929,390.500000,222.785714,5,0.339127,0.391942,0.295986,0.019948,0.229311,-0.332529,0.337417,0.325317,0.353755,0.003025,159.414727,0.321185,0.045961,0.328880,Cell 1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
819,398.714286,374.100000,277.442857,820,0.341419,0.398684,0.278371,0.022494,-0.616688,0.706583,0.345748,0.330875,0.354459,0.001891,147.987523,0.281368,0.017759,0.219416,Cell 6
820,400.057143,372.114286,266.342857,821,0.335910,0.365674,0.293998,0.016570,-0.379847,-0.189990,0.337066,0.328188,0.348624,0.001891,139.261591,0.245965,0.026495,0.226925,Cell 6
821,401.348000,400.024000,264.264000,822,0.373556,0.435755,0.284875,0.026402,-0.337575,-0.132086,0.374776,0.355870,0.392840,0.006752,235.890315,0.721316,0.029210,0.130676,Cell 6
822,403.783019,404.886792,233.028302,823,0.348780,0.397968,0.298770,0.019457,0.059488,-0.191630,0.350133,0.334463,0.359760,0.002863,165.552006,0.360844,0.026434,0.265593,Cell 6


In [176]:
UnnormNSsavepath = "D:/Downloads/Drug Stimulated INS1E Cells/CSV_files_revisions/Unnorm"
Unnorm_Ves_df["Condition"] = "No Stim"
Unnorm_Ves_df.to_csv(UnnormNSsavepath + '/Unorm_NS_vesicles.csv', index = False)

In [518]:
#-------------------------------------------
#Raw Distances Analysis
Raw_Unnorm_VesNS_7_22 = NS_7_22_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesNS_6_5 = NS_6_5_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesNS_1537_16_18 = NS_1537_16_18_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesNS_7_8_9 = NS_7_8_9_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesNS_6_17_19 = NS_6_17_19_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesNS_1537_19 = NS_1537_19_class.Ves_Analysis_Raw_Unorm()

#Assigning cell labels
cell1 = "Cell 1"
cell2 = "Cell 2"
cell3 = "Cell 3"
cell4 = "Cell 4"
cell5 = "Cell 5"
cell6 = "Cell 6"
Raw_Unnorm_VesNS_7_22["Cell"] = cell1
Raw_Unnorm_VesNS_6_5["Cell"] = cell2 
Raw_Unnorm_VesNS_1537_16_18["Cell"] = cell3
Raw_Unnorm_VesNS_7_8_9["Cell"] = cell4 
Raw_Unnorm_VesNS_6_17_19["Cell"] = cell5 
Raw_Unnorm_VesNS_1537_19["Cell"] = cell6 

#Combine raw distance ves files
Raw_Unnorm_Ves_df = pd.concat([Raw_Unnorm_VesNS_7_22, Raw_Unnorm_VesNS_6_5, Raw_Unnorm_VesNS_1537_16_18, 
                           Raw_Unnorm_VesNS_7_8_9, Raw_Unnorm_VesNS_6_17_19, Raw_Unnorm_VesNS_1537_19])

UnnormNS_Rawsavepath = "D:/Downloads/Drug Stimulated INS1E Cells/CSV_files/Raw_Dist"
Raw_Unnorm_Ves_df.to_csv(UnnormNS_Rawsavepath + '/Raw_Unorm_NS_vesicles.csv', index = False)

C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang

In [519]:
UnnormNS_Rawsavepath = "D:/Downloads/Drug Stimulated INS1E Cells/CSV_files/Raw_Dist"
Raw_Unnorm_Ves_df["Condition"] = "No Stim"
Raw_Unnorm_Ves_df.to_csv(UnnormNS_Rawsavepath + '/Raw_Unorm_NS_vesicles.csv', index = False)

**Glucose Analysis** 

In [7]:
#Glucose classes 

Glu_9905_7_class = INS1E(Glu_9905_7_files, LACfactordict['9905_7'])
Glu_9905_8_class = INS1E(Glu_9905_8_files, LACfactordict['9905_8'])
Glu_9905_11_12_class = INS1E(Glu_9905_11_12_files, LACfactordict['9905_11-12'])
Glu_9905_8_9_class = INS1E(Glu_9905_8_9_files, LACfactordict['9905_8'])
Glu_9908_4_5_class = INS1E(Glu_9908_4_5_files, LACfactordict['9908_4-5'])
Glu_9905_12_13_class = INS1E(Glu_9905_12_13_files, LACfactordict['9905_11-12'])

In [178]:
#Glucose 30 min analysis

Unnorm_VesGlu_9905_7 = Glu_9905_7_class.Ves_Analysis_unnorm()
Unnorm_VesGlu_9905_8 = Glu_9905_8_class.Ves_Analysis_unnorm()
Unnorm_VesGlu_9905_11_12 = Glu_9905_11_12_class.Ves_Analysis_unnorm()
Unnorm_VesGlu_9905_8_9 = Glu_9905_8_9_class.Ves_Analysis_unnorm()
Unnorm_VesGlu_9908_4_5 = Glu_9908_4_5_class.Ves_Analysis_unnorm()
Unnorm_VesGlu_9905_12_13 = Glu_9905_12_13_class.Ves_Analysis_unnorm()

C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:206: RuntimeWarning: Map ID string not found - not an MRC file, or file is corrupt
  warnings.warn(msg, RuntimeWarning)
C:\Users\kchang42\AppData\Local\Temp\ipykernel_6992\1540841857.py:128: RuntimeWarning: invalid value encountered in divide
  Mem_dist_norm = Mem_dist / (Nuc_dist + Mem_dist)
C:\Users\kchang42\AppData\Loc

In [179]:
Unnorm_VesGlu_9905_7["Cell"] = "Cell 7"
Unnorm_VesGlu_9905_8["Cell"] = "Cell 8" 
Unnorm_VesGlu_9905_11_12["Cell"] = "Cell 9" 
Unnorm_VesGlu_9905_8_9["Cell"] = "Cell 10"
Unnorm_VesGlu_9908_4_5["Cell"] = "Cell 11"
Unnorm_VesGlu_9905_12_13["Cell"] = "Cell 12" 

Unnorm_Ves_Glu_df = pd.concat([Unnorm_VesGlu_9905_7, Unnorm_VesGlu_9905_8, Unnorm_VesGlu_9905_11_12, Unnorm_VesGlu_9905_8_9, 
                           Unnorm_VesGlu_9908_4_5, Unnorm_VesGlu_9905_12_13])
Unnorm_Ves_Glu_df

,Centroid X,Centroid Y,Centroid Z,Group,LAC Value,LAC Max,LAC Min,LAC Std Dev,LAC Skew,LAC Kurtosis,LAC Median,LAC 25th Quantile,LAC 75th Quantile,Volume (um3),Geometric Diameter (nm),Ellipsoid Surface Area (um2),Min Distance to PM EDT,Mito Dist EDT,Cell
0,81.671875,234.656250,220.406250,1,0.345842,0.393650,0.305841,0.021504,0.456026,-0.486163,0.340801,0.331901,0.362724,0.001799,108.145129,0.153230,0.110687,0.291627,Cell 7
1,84.155172,241.034483,210.000000,2,0.310798,0.367288,0.271253,0.020862,0.239564,-0.038784,0.311241,0.296518,0.323770,0.001630,131.008494,0.217523,0.059405,0.315800,Cell 7
2,85.607143,243.821429,253.160714,3,0.379432,0.413656,0.334049,0.017529,0.026026,-0.466432,0.377025,0.366499,0.392367,0.001574,119.029116,0.184425,0.058716,0.258986,Cell 7
3,85.000000,257.769231,230.500000,4,0.352274,0.382171,0.325958,0.014455,0.380256,-0.571054,0.350869,0.342238,0.359739,0.000731,69.608650,0.062553,0.097490,0.287775,Cell 7
4,86.809524,219.071429,240.476190,5,0.382397,0.433755,0.335053,0.020683,0.336051,-0.246396,0.382890,0.367657,0.396566,0.002361,131.008494,0.217523,0.055107,0.261469,Cell 7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
512,352.802469,189.777778,256.049383,513,0.303010,0.346693,0.267152,0.017148,0.371583,-0.324349,0.303692,0.290429,0.313321,0.002277,131.008494,0.217523,0.369751,0.048376,Cell 12
513,351.862069,253.620690,261.241379,514,0.294144,0.326161,0.266853,0.014205,-0.055105,-0.730946,0.294098,0.284427,0.305701,0.001630,131.008494,0.217523,0.056339,0.194601,Cell 12
514,353.581081,206.256757,226.729730,515,0.265486,0.292437,0.240434,0.011676,-0.256746,-0.320137,0.266648,0.259426,0.273818,0.002080,141.124622,0.252590,0.100139,0.112037,Cell 12
515,356.852273,214.397727,267.454545,516,0.314405,0.338402,0.282448,0.011799,-0.444917,0.024376,0.315653,0.306494,0.322768,0.002473,152.021891,0.290417,0.068853,0.226279,Cell 12


In [180]:
UnnormGlusavepath = "D:/Downloads/Drug Stimulated INS1E Cells/CSV_files_revisions/Unnorm"
Unnorm_Ves_Glu_df["Condition"] = "Glucose"
Unnorm_Ves_Glu_df.to_csv(UnnormGlusavepath + '/Unorm_Glu_vesicles.csv', index = False)

In [513]:
#----------------------------------------------
#Raw Distances Analysis

Raw_Unnorm_VesGlu_9905_7 = Glu_9905_7_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesGlu_9905_8 = Glu_9905_8_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesGlu_9905_11_12 = Glu_9905_11_12_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesGlu_9905_8_9 = Glu_9905_8_9_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesGlu_9908_4_5 = Glu_9908_4_5_class.Ves_Analysis_Raw_Unorm()
Raw_Unnorm_VesGlu_9905_12_13 = Glu_9905_12_13_class.Ves_Analysis_Raw_Unorm()

#Assigning cell labels
Raw_Unnorm_VesGlu_9905_7["Cell"] = "Cell 7"
Raw_Unnorm_VesGlu_9905_8["Cell"] = "Cell 8" 
Raw_Unnorm_VesGlu_9905_11_12["Cell"] = "Cell 9" 
Raw_Unnorm_VesGlu_9905_8_9["Cell"] = "Cell 10"
Raw_Unnorm_VesGlu_9908_4_5["Cell"] = "Cell 11" 
Raw_Unnorm_VesGlu_9905_12_13["Cell"] = "Cell 12" 

#Combine raw distance ves files
Raw_Unnorm_VesGlu_df = pd.concat([Raw_Unnorm_VesGlu_9905_7, Raw_Unnorm_VesGlu_9905_8, Raw_Unnorm_VesGlu_9905_11_12, Raw_Unnorm_VesGlu_9905_8_9, 
                           Raw_Unnorm_VesGlu_9908_4_5, Raw_Unnorm_VesGlu_9905_12_13])

Raw_UnnormGlusavepath = "D:/Downloads/Drug Stimulated INS1E Cells/CSV_files/Raw_Dist"
Raw_Unnorm_VesGlu_df["Condition"] = "Glucose"
Raw_Unnorm_VesGlu_df.to_csv(Raw_UnnormGlusavepath + '/Raw_Unorm_Glu_vesicles.csv', index = False)

C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang42\AppData\Local\anaconda3\Lib\site-packages\mrcfile\mrcinterpreter.py:216: RuntimeWarning: Unrecognised machine stamp: 0x00 0x00 0x00 0x00
  warnings.warn(str(err), RuntimeWarning)
C:\Users\kchang